In [1]:
import pandas as pd
import numpy as np
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# 读取预处理后的数据
land_data = pd.read_csv('land_data_clean.csv')
crop_data = pd.read_csv('crop_data_clean.csv')
stats_2023 = pd.read_csv('stats_2023_clean.csv')
planting_2023 = pd.read_csv('planting_2023.csv')

print("=== 数据列名检查 ===")
print("planting_2023列名:", planting_2023.columns.tolist())
print("stats_2023列名:", stats_2023.columns.tolist())

# 建立基础数据架构
print("\n=== 建立数学模型基础架构 ===")

# 1. 定义关键参数
years = list(range(2024, 2031))  # 2024-2030年
seasons = ['单季', '第一季', '第二季']

# 2. 地块类型与作物兼容性
land_types = ['平旱地', '梯田', '山坡地', '水浇地', '普通大棚', '智慧大棚']
crop_types = ['粮食', '粮食（豆类）', '蔬菜', '蔬菜（豆类）', '食用菌']

# 创建兼容性矩阵
compatibility = {
    '平旱地': ['粮食', '粮食（豆类）'],
    '梯田': ['粮食', '粮食（豆类）'],
    '山坡地': ['粮食', '粮食（豆类）'],
    '水浇地': ['粮食', '粮食（豆类）', '蔬菜', '蔬菜（豆类）'],
    '普通大棚': ['蔬菜', '蔬菜（豆类）', '食用菌'],
    '智慧大棚': ['蔬菜', '蔬菜（豆类）']
}

# 3. 季次约束
season_constraints = {
    '平旱地': ['单季'],
    '梯田': ['单季'],
    '山坡地': ['单季'],
    '水浇地': ['单季', '第一季', '第二季'],
    '普通大棚': ['第一季', '第二季'],
    '智慧大棚': ['第一季', '第二季']
}

# 4. 获取2023年预期销售量（根据实际种植面积和亩产量估算）
def calculate_expected_sales():
    # 先获取每个地块的类型
    land_type_mapping = land_data.set_index('地块名称')['地块类型'].to_dict()
    
    # 将地块类型添加到种植数据中
    planting_with_type = planting_2023.copy()
    planting_with_type['地块类型'] = planting_with_type['地块名称'].map(land_type_mapping)
    
    # 合并数据
    planting_merged = planting_with_type.merge(stats_2023, 
                                         on=['作物编号', '地块类型', '种植季次'], 
                                         how='left')
    
    # 计算总产量
    planting_merged['总产量'] = planting_merged['种植面积/亩'] * planting_merged['亩产量/斤']
    
    # 按作物编号聚合总产量
    expected_sales = planting_merged.groupby('作物编号')['总产量'].sum().reset_index()
    expected_sales = expected_sales.merge(crop_data[['作物编号', '作物名称']], on='作物编号')
    expected_sales.columns = ['作物编号', '预期销售量/斤', '作物名称']
    
    return expected_sales

expected_sales_2023 = calculate_expected_sales()

print("\n=== 2023年预期销售量 ===")
print(expected_sales_2023.head(10))

# 5. 获取作物基础信息
def get_crop_base_info():
    crop_info = {}
    for _, row in crop_data.iterrows():
        crop_id = row['作物编号']
        crop_name = row['作物名称']
        crop_type = row['作物类型']
        
        # 获取亩产量、种植成本、销售单价
        crop_stats = stats_2023[stats_2023['作物编号'] == crop_id]
        
        if not crop_stats.empty:
            avg_yield = crop_stats['亩产量/斤'].mean()
            avg_cost = crop_stats['种植成本/(元/亩)'].mean()
            avg_price = crop_stats['销售单价_平均'].mean()
            
            crop_info[crop_id] = {
                'name': crop_name,
                'type': crop_type,
                'yield_per_mu': avg_yield,
                'cost_per_mu': avg_cost,
                'price_per_jin': avg_price
            }
    
    return crop_info

crop_base_info = get_crop_base_info()

print("\n=== 作物基础信息 ===")
for crop_id, info in list(crop_base_info.items())[:5]:
    print(f"作物{crop_id}({info['name']}): 亩产{info['yield_per_mu']:.0f}斤, 成本{info['cost_per_mu']}元/亩, 单价{info['price_per_jin']:.2f}元/斤")

# 6. 建立模型参数
print("\n=== 模型参数 ===")
print(f"规划年份: {years}")
print(f"地块总数: {len(land_data)}")
print(f"作物总数: {len(crop_base_info)}")
print(f"地块类型: {land_types}")

# 保存基础数据
expected_sales_2023.to_csv('expected_sales_2023.csv', index=False, encoding='utf-8-sig')

# 将作物基础信息转换为DataFrame
crop_info_df = pd.DataFrame.from_dict(crop_base_info, orient='index').reset_index()
crop_info_df.columns = ['作物编号', '作物名称', '作物类型', '亩产量/斤', '种植成本/(元/亩)', '销售单价/(元/斤)']
crop_info_df.to_csv('crop_base_info.csv', index=False, encoding='utf-8-sig')

print("\n=== 数据准备完成 ===")
print("已为问题1的模型求解做好准备")

=== 数据列名检查 ===
planting_2023列名: ['地块名称', '作物编号', '作物名称', '作物类型', '种植面积/亩', '种植季次']
stats_2023列名: ['序号', '作物编号', '作物名称', '地块类型', '种植季次', '亩产量/斤', '种植成本/(元/亩)', '销售单价/(元/斤)', '销售单价_平均']

=== 建立数学模型基础架构 ===

=== 2023年预期销售量 ===
   作物编号   预期销售量/斤 作物名称
0     1   57000.0   黄豆
1     2   21850.0   黑豆
2     3   22400.0   红豆
3     4   33040.0   绿豆
4     5    9875.0   爬豆
5     6  170840.0   小麦
6     7  132750.0   玉米
7     8   71400.0   谷子
8     9   30000.0   高粱
9    10   12500.0   黍子

=== 作物基础信息 ===
作物1(黄豆): 亩产380斤, 成本400.0元/亩, 单价3.25元/斤
作物2(黑豆): 亩产475斤, 成本400.0元/亩, 单价7.50元/斤
作物3(红豆): 亩产380斤, 成本350.0元/亩, 单价8.25元/斤
作物4(绿豆): 亩产332斤, 成本350.0元/亩, 单价7.00元/斤
作物5(爬豆): 亩产395斤, 成本350.0元/亩, 单价6.75元/斤

=== 模型参数 ===
规划年份: [2024, 2025, 2026, 2027, 2028, 2029, 2030]
地块总数: 54
作物总数: 41
地块类型: ['平旱地', '梯田', '山坡地', '水浇地', '普通大棚', '智慧大棚']

=== 数据准备完成 ===
已为问题1的模型求解做好准备
